In [1]:
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

DB_PATH = (
    Path.cwd()
    / "projects"
    / "P0_Data_Platform"
    / "datasets"
    / "lendingclub"
    / "data"
    / "warehouse"
    / "duckdb"
    / "lendingclub.duckdb"
)

print(DB_PATH)
conn = duckdb.connect(str(DB_PATH))

d:\project_lighthouse\projects\P0_Data_Platform\datasets\lendingclub\data\warehouse\duckdb\lendingclub.duckdb


In [2]:
# Mart Row Count
conn.execute('''
select count(*) as mart_rows
from mart.geographic_performance
''').fetchdf()

,mart_rows
0,51


In [3]:
# Geographic Grain Validation
conn.execute('''
select
    count(*) as rows,
    count(distinct addr_state) as unique_states
from mart.geographic_performance
''').fetchdf()

,rows,unique_states
0,51,51


In [4]:
# Loan Count Reconciliation
conn.execute('''
select
(
    select count(*)
    from feature.loan_features_v1
    where addr_state is not null
) as feature_rows,
(
    select sum(loan_count)
    from mart.geographic_performance
) as mart_loan_count
''').fetchdf()

,feature_rows,mart_loan_count
0,2260668,2260668.0


In [5]:
# Table Structure
conn.execute('''
describe mart.geographic_performance
''').fetchdf()

,column_name,column_type,null,key,default,extra
0,addr_state,VARCHAR,YES,None,None,None
1,loan_count,BIGINT,YES,None,None,None
2,total_loan_amount,DOUBLE,YES,None,None,None
3,avg_loan_amount,DOUBLE,YES,None,None,None
4,avg_interest_rate,DOUBLE,YES,None,None,None
5,avg_fico,DOUBLE,YES,None,None,None
6,avg_dti,DOUBLE,YES,None,None,None
7,avg_credit_history_years,DOUBLE,YES,None,None,None
8,avg_loan_to_income_ratio,DOUBLE,YES,None,None,None
9,avg_revolving_utilization,DOUBLE,YES,None,None,None


In [6]:
# Default Rate Reconciliation
conn.execute('''
select
(
    select round(avg(default_flag) * 100, 4)
    from feature.loan_features_v1
    where addr_state is not null
) as feature_default_rate,
(
    select round(sum(default_count) * 100.0 / sum(loan_count), 4)
    from mart.geographic_performance
) as mart_default_rate
''').fetchdf()

,feature_default_rate,mart_default_rate
0,12.8646,12.8646


In [7]:
# Top States By Volume
conn.execute('''
select *
from mart.geographic_performance
order by loan_count desc
limit 20
''').fetchdf()

,addr_state,loan_count,total_loan_amount,avg_loan_amount,avg_interest_rate,avg_fico,avg_dti,avg_credit_history_years,avg_loan_to_income_ratio,avg_revolving_utilization,default_count,default_rate,high_utilization_count,high_utilization_rate
0,CA,314533,4.808480e+09,15287.680784,12.976528,700.196973,17.233905,16.096507,0.659533,50.172403,41641.0,13.24,43660.0,13.88
1,NY,186389,2.767161e+09,14846.158840,13.260060,701.221244,16.836901,15.929773,0.401724,49.623233,26325.0,14.12,24731.0,13.27
2,TX,186335,2.931134e+09,15730.450667,12.999444,701.462068,19.894080,16.481133,0.758165,49.329873,23759.0,12.75,22710.0,12.19
3,FL,161991,2.333034e+09,14402.247656,13.164735,699.128467,18.933998,15.971532,0.957966,47.922054,22447.0,13.86,17967.0,11.09
4,IL,91173,1.410452e+09,15470.061860,12.960124,700.837847,18.739347,16.676931,0.495984,50.567034,10071.0,11.05,12666.0,13.89
5,NJ,83132,1.316208e+09,15832.747618,12.989854,700.748412,17.349175,16.678215,0.212718,49.081698,11031.0,13.27,10435.0,12.55
6,PA,76939,1.131988e+09,14712.797801,13.145725,701.245116,19.716737,16.734269,0.672168,50.726141,10214.0,13.28,10817.0,14.06
7,OH,75132,1.077143e+09,14336.673122,13.137272,700.533994,20.249965,16.677955,0.241839,50.009126,9564.0,12.73,9657.0,12.85
8,GA,74196,1.136791e+09,15321.457019,13.208650,699.715712,19.261108,16.546947,0.231681,50.331111,8687.0,11.71,9987.0,13.46
9,VA,62954,1.013016e+09,16091.366315,13.113697,701.187375,19.054895,16.627304,0.643492,52.849394,8208.0,13.04,10719.0,17.03


In [8]:
# Highest Default Rate States
conn.execute('''
select *
from mart.geographic_performance
where loan_count >= 1000
order by default_rate desc
limit 20
''').fetchdf()

,addr_state,loan_count,total_loan_amount,avg_loan_amount,avg_interest_rate,avg_fico,avg_dti,avg_credit_history_years,avg_loan_to_income_ratio,avg_revolving_utilization,default_count,default_rate,high_utilization_count,high_utilization_rate
0,AL,27284,4.006992e+08,14686.235523,13.567154,700.077463,20.635317,16.980290,0.750073,51.641742,4235.0,15.52,4095.0,15.01
1,MS,12639,1.864272e+08,14750.150328,13.445814,699.236490,20.964321,17.184696,1.428379,50.326188,1929.0,15.26,1785.0,14.12
2,AR,17074,2.406505e+08,14094.558979,13.354703,701.348161,21.044830,16.474316,0.243767,50.068048,2602.0,15.24,2238.0,13.11
3,OK,20691,3.106654e+08,15014.517181,13.263297,701.663694,20.620884,16.428058,1.938400,52.490947,3104.0,15.00,3374.0,16.31
4,LA,25759,3.820463e+08,14831.564696,13.224466,702.144338,19.950426,16.507962,0.231264,50.572218,3851.0,14.95,3758.0,14.59
5,NV,32657,4.701653e+08,14397.075665,13.205397,698.385369,19.105376,15.921567,0.232332,48.764396,4744.0,14.53,3976.0,12.18
6,NY,186389,2.767161e+09,14846.158840,13.260060,701.221244,16.836901,15.929773,0.401724,49.623233,26325.0,14.12,24731.0,13.27
7,NM,11986,1.782805e+08,14874.063491,13.155975,700.971467,20.958039,16.821654,0.406274,51.107761,1690.0,14.10,1726.0,14.40
8,FL,161991,2.333034e+09,14402.247656,13.164735,699.128467,18.933998,15.971532,0.957966,47.922054,22447.0,13.86,17967.0,11.09
9,HI,10668,1.695621e+08,15894.460067,13.786782,697.980643,19.569023,16.286618,0.249504,56.275922,1474.0,13.82,2326.0,21.80


In [ ]:
conn.close()

: 